# PlayReactant — Benchmark Sweep on Google Colab TPU/GPU

Resolution sweep for all 4 physics solvers (Stokes 2D/3D, PorowavesStokes 2D/3D)
using [Reactant.jl](https://github.com/EnzymeAD/Reactant.jl).

**Before running:** set *Runtime → Change runtime type* to **TPU v2-8** or **GPU (T4)**.

The Colab Julia 1.12 image already includes: `Reactant`, `Makie`, `Plots`, `CSV`, `DataFrames`.
We only add the missing pieces (`CairoMakie`, `TOML`) and clone the repo.

In [ ]:
# ── 1. Add missing packages ──────────────────────────────────────────
# Reactant, Makie, Plots, CSV, DataFrames are pre-installed in the Colab image.
# We only need CairoMakie (off-screen PNG) and TOML (stdlib, just ensure available).
using Pkg
for pkg in ["CairoMakie"]
    if !haskey(Pkg.project().dependencies, pkg)
        Pkg.add(pkg)
    end
end
println("Environment ready")

In [ ]:
# ── 2. Clone / update the PlayReactant repo ──────────────────────────
REPO_URL = "https://github.com/lraess/PlayReactant.git"  # ← update if needed
REPO_DIR = "/content/PlayReactant"

if !isdir(REPO_DIR)
    run(`git clone $REPO_URL $REPO_DIR`)
else
    run(`git -C $REPO_DIR pull`)
end

# Verify key scripts are present
for s in ["Stokes_react2D.jl", "Stokes_react3D.jl",
          "PorowavesStokes_react2D.jl", "PorowavesStokes_react3D.jl",
          "bench_sweep.jl"]
    @assert isfile(joinpath(REPO_DIR, "egu26", s)) "Missing: $s"
end
println("Repo ready at $REPO_DIR ✓")

In [ ]:
# ── 3. Configure and run the sweep ───────────────────────────────────
# Detect backend: prefer TPU, fall back to GPU, then CPU.
backend = if isdir("/dev/accel0")
    "tpu"
elseif success(`nvidia-smi`)
    "gpu"
else
    "cpu"
end
@info "Using Reactant backend: $backend"

# Resolutions — smaller than HPC defaults to fit Colab memory.
ENV["BENCH_BACKEND"]    = backend
ENV["BENCH_VIZ"]        = "1"
ENV["BENCH_SWEEP_OUT"]  = "/content/bench_sweep_results.toml"
ENV["BENCH_RES_S2D"]    = "64,128,256,512"
ENV["BENCH_RES_S3D"]    = "32,64,128"
ENV["BENCH_RES_PW2D"]   = "32,64,128,256"
ENV["BENCH_RES_PW3D"]   = "16,32,64"

# The _bench_sweep flag suppresses the hardcoded main() call at the bottom
# of each included script; must be defined in Main before any include.
const _bench_sweep = true

cd(joinpath("/content/PlayReactant", "egu26")) do
    include("bench_sweep.jl")
end

In [ ]:
# ── 4. Print results table ───────────────────────────────────────────
using TOML, Printf
data = TOML.parsefile("/content/bench_sweep_results.toml")
runs = data["runs"]
println("Backend: ", data["backend"], "\n")
@printf "%-20s %6s %12s %10s %8s %10s\n" "solver" "nx" "t_compile[s]" "t_run[s]" "niter" "T_eff[GB/s]"
println("-"^70)
for r in runs
    @printf "%-20s %6d %12.2f %10.3f %8d %10.2f\n" r["solver"] r["nx"] r["t_compile"] r["t_run"] r["niter"] r["T_eff"]
end

In [ ]:
# ── 5. Plot: t_run, niter, T_eff vs nx; min compile time bar chart ───
using CairoMakie, TOML

data = TOML.parsefile("/content/bench_sweep_results.toml")
runs = data["runs"]

solvers = unique(r["solver"] for r in runs)
colors  = Makie.wong_colors()
markers = [:circle, :rect, :diamond, :utriangle]
sol_col = Dict(s => colors[i]  for (i,s) in enumerate(solvers))
sol_mrk = Dict(s => markers[i] for (i,s) in enumerate(solvers))

fig = Figure(; size=(1000, 700))
ax_trun  = Axis(fig[1,1]; xlabel="nx", ylabel="t_run [s]",    title="Wall time",
                xscale=log2, yscale=log10)
ax_teff  = Axis(fig[1,2]; xlabel="nx", ylabel="T_eff [GB/s]", title="Effective memory throughput",
                xscale=log2)
ax_niter = Axis(fig[2,1]; xlabel="nx", ylabel="niter",        title="Iterations to convergence",
                xscale=log2)
ax_comp  = Axis(fig[2,2]; xlabel="solver", ylabel="min t_compile [s]",
                title="Min compile time per solver")

# min compile per solver
compile_mins = Dict(s => minimum(r["t_compile"] for r in runs if r["solver"]==s) for s in solvers)

for s in solvers
    sr   = filter(r -> r["solver"] == s, runs)
    nxs  = [r["nx"]        for r in sr]
    idxs = sortperm(nxs)
    nxs  = nxs[idxs]
    trun = [r["t_run"]     for r in sr][idxs]
    teff = [r["T_eff"]     for r in sr][idxs]
    nit  = [Float64(r["niter"]) for r in sr][idxs]
    scatterlines!(ax_trun,  nxs, trun; color=sol_col[s], marker=sol_mrk[s], label=s)
    scatterlines!(ax_teff,  nxs, teff; color=sol_col[s], marker=sol_mrk[s], label=s)
    scatterlines!(ax_niter, nxs, nit;  color=sol_col[s], marker=sol_mrk[s], label=s)
end
Legend(fig[1,3], ax_trun; framevisible=false)

# compile bar chart
barplot!(ax_comp, 1:length(solvers), [compile_mins[s] for s in solvers];
         color=[sol_col[s] for s in solvers])
ax_comp.xticks = (1:length(solvers), solvers)

save("/content/bench_sweep_results.png", fig)
display(fig)

In [ ]:
# ── 6. Download results to local machine ─────────────────────────────
from google.colab import files
files.download("/content/bench_sweep_results.toml")
files.download("/content/bench_sweep_results.png")